In [3]:
!nvidia-smi

Thu Aug 13 05:48:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.1 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import json

dataset_path = '/content/drive/MyDrive/day28_full_dataset.jsonl'
# (apna actual path yahan use karo agar different hai)

with open(dataset_path, 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

print(f"✓ Total pairs loaded: {len(data)}")
print(f"\nFirst entry preview:")
print(json.dumps(data[0], ensure_ascii=False, indent=2))

✓ Total pairs loaded: 200

First entry preview:
{
  "messages": [
    {
      "role": "system",
      "content": "Tum ek Hinglish AI/ML tutor ho jo hamesha pehle ek everyday analogy deta hai, phir uske baad technical explanation deta hai. Tumhara tone friendly aur encouraging hota hai."
    },
    {
      "role": "user",
      "content": "TF-IDF kya hota hai aur ye simple word counting se better kyun hai?"
    },
    {
      "role": "assistant",
      "content": "Socho tumhare paas 100 news articles hain aur tum dhoondna chahte ho ki kaunse words important hain. Agar tum sirf counting karo, to 'the', 'is', 'a' jaise common words sabse zyada baar aayenge — lekin woh useless hain. TF-IDF do cheezein combine karta hai: TF (Term Frequency) batata hai ek word document mein kitni baar aaya, aur IDF (Inverse Document Frequency) batata hai woh word kitna 'rare' hai across saare documents. Jo word ek document mein bohot use hua ho lekin baaki documents mein rare ho — wahi asal mein important ho

In [7]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/day28_finetune/checkpoints'
LOCAL_OUTPUT_DIR = '/content/local_training_output'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

print(f"✓ Checkpoint folder ready: {CHECKPOINT_DIR}")

✓ Checkpoint folder ready: /content/drive/MyDrive/day28_finetune/checkpoints


In [8]:
from transformers import TrainerCallback
import shutil

class DriveCheckpointCallback(TrainerCallback):
    """Har N steps pe local checkpoint ko Drive pe sync karta hai."""

    def __init__(self, save_every_n_steps=50, local_dir=LOCAL_OUTPUT_DIR, drive_dir=CHECKPOINT_DIR):
        self.save_every_n_steps = save_every_n_steps
        self.local_dir = local_dir
        self.drive_dir = drive_dir

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and state.global_step % self.save_every_n_steps == 0:
            local_ckpt = os.path.join(self.local_dir, f"checkpoint-{state.global_step}")
            drive_ckpt = os.path.join(self.drive_dir, f"checkpoint-{state.global_step}")
            if os.path.exists(local_ckpt):
                shutil.copytree(local_ckpt, drive_ckpt, dirs_exist_ok=True)
                print(f"✓ Step {state.global_step}: checkpoint synced to Drive")
        return control

print("✓ DriveCheckpointCallback defined")

✓ DriveCheckpointCallback defined


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

sample = data[0]  # tumhara TF-IDF wala pair
formatted_text = tokenizer.apply_chat_template(
    sample["messages"], tokenize=False, add_generation_prompt=False
)
print("Formatted (raw string) version:\n")
print(formatted_text)

token_ids = tokenizer.apply_chat_template(sample["messages"], tokenize=True)
print(f"\nToken count for this example: {len(token_ids)}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Formatted (raw string) version:

<|im_start|>system
Tum ek Hinglish AI/ML tutor ho jo hamesha pehle ek everyday analogy deta hai, phir uske baad technical explanation deta hai. Tumhara tone friendly aur encouraging hota hai.<|im_end|>
<|im_start|>user
TF-IDF kya hota hai aur ye simple word counting se better kyun hai?<|im_end|>
<|im_start|>assistant
Socho tumhare paas 100 news articles hain aur tum dhoondna chahte ho ki kaunse words important hain. Agar tum sirf counting karo, to 'the', 'is', 'a' jaise common words sabse zyada baar aayenge — lekin woh useless hain. TF-IDF do cheezein combine karta hai: TF (Term Frequency) batata hai ek word document mein kitni baar aaya, aur IDF (Inverse Document Frequency) batata hai woh word kitna 'rare' hai across saare documents. Jo word ek document mein bohot use hua ho lekin baaki documents mein rare ho — wahi asal mein important hota hai. Isliye TF-IDF simple counting se zyada smart hai.<|im_end|>


Token count for this example: 2


In [10]:
def validate_chatml_dataset(data):
    issues = {"empty_fields": [], "duplicates": [], "length_outliers": []}
    seen = set()
    lengths = []

    for i, row in enumerate(data):
        msgs = row.get("messages", [])
        user_msg = next((m["content"] for m in msgs if m["role"] == "user"), "")
        assistant_msg = next((m["content"] for m in msgs if m["role"] == "assistant"), "")

        if not user_msg.strip() or not assistant_msg.strip():
            issues["empty_fields"].append(i)

        key = (user_msg.strip().lower(), assistant_msg.strip().lower())
        if key in seen:
            issues["duplicates"].append(i)
        seen.add(key)

        lengths.append(len(assistant_msg.split()))

    avg = sum(lengths) / len(lengths)
    std = (sum((x - avg) ** 2 for x in lengths) / len(lengths)) ** 0.5
    for i, l in enumerate(lengths):
        if abs(l - avg) > 2 * std:
            issues["length_outliers"].append((i, l))

    print(f"Total rows: {len(data)}")
    print(f"Empty fields: {len(issues['empty_fields'])}")
    print(f"Duplicates: {len(issues['duplicates'])}")
    print(f"Avg answer length: {avg:.1f} words (std: {std:.1f})")
    print(f"Length outliers: {len(issues['length_outliers'])}")
    return issues

issues = validate_chatml_dataset(data)

Total rows: 200
Empty fields: 0
Duplicates: 0
Avg answer length: 88.1 words (std: 9.3)
Length outliers: 13


In [11]:
from huggingface_hub import login
login()

In [12]:
from datasets import Dataset

# Tumhare 200 pairs ko HuggingFace Dataset format mein convert kar rahe hain
hf_dataset = Dataset.from_list(data)

# Tumhara username "nooruiit-864" hai, isliye:
REPO_NAME = "nooruiit-864/hinglish-ai-ml-tutor-dataset"

hf_dataset.push_to_hub(REPO_NAME)
print(f"✓ Dataset pushed to: https://huggingface.co/datasets/{REPO_NAME}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp3w0sy7l3.parquet    : 100%|##########|  166kB /  166kB            

/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:11610: UserWarning: Warnings while validating metadata in README.md:
- The task_categories "conversational" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, text-to-video, image-text-to-text, image-text-to-image, image-te

✓ Dataset pushed to: https://huggingface.co/datasets/nooruiit-864/hinglish-ai-ml-tutor-dataset


In [13]:
from huggingface_hub import HfApi

data_card = """---
language:
- hi
- en
license: mit
task_categories:
- text-generation
- conversational
tags:
- instruction-tuning
- chatml
- hinglish
- ai-ml-education
size_categories:
- n<1K
---

# Hinglish AI/ML Tutor Dataset

## Dataset Description
A hand-curated instruction-tuning dataset of 200 Q&A pairs covering AI/ML engineering
concepts (tokenization, embeddings, transformers, RAG, LoRA/QLoRA, STT/TTS, deployment,
and web security). Every answer follows a consistent "Hinglish tutor" persona: an
everyday analogy first, followed by the technical explanation.

## Format
ChatML format (`messages` field with `system`/`user`/`assistant` roles), one JSON object per line (JSONL).

## Source
Hand-curated from a 42-day AI Engineer Internship curriculum, covering topics from
NLP fundamentals through production RAG and fine-tuning pipelines.

## Intended Use
LoRA/QLoRA fine-tuning of a small instruction-tuned model to produce a
Hinglish-speaking AI/ML tutor persona.

## Size
200 examples, ~180KB.
"""

api = HfApi()
api.upload_file(
    path_or_fileobj=data_card.encode(),
    path_in_repo="README.md",
    repo_id=REPO_NAME,
    repo_type="dataset"
)
print("✓ Data card uploaded")

✓ Data card uploaded


In [14]:
repo_id = "nooruiit-864/Finetunedataset"  # pick a descriptive slug for your domain

hf_dataset.push_to_hub(repo_id, private=False)  # set private=True if you'd rather not make it public

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpxz3y44r0.parquet    : 100%|##########|  166kB /  166kB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/nooruiit-864/Finetunedataset/commit/f23d716a3c6341df5d078697e7020675725ae22f', commit_message='Upload dataset', commit_description='', oid='f23d716a3c6341df5d078697e7020675725ae22f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/nooruiit-864/Finetunedataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='nooruiit-864/Finetunedataset'), pr_revision=None, pr_num=None)

In [15]:
# ==== EDIT THESE ====
USE_LOCAL_FILE    = True   # True = load from uploaded .jsonl in this Colab session
                            # False = load from HF Hub (requires DATASET_REPO_ID below)

LOCAL_JSONL_PATH  = "day28_full_dataset.jsonl"   # upload this file into Colab's file browser (left sidebar) first
DATASET_REPO_ID   = "nooruiit-864/YOUR-DATASET-NAME"   # only used if USE_LOCAL_FILE = False

# ---- Swap this line to try a different base model ----
BASE_MODEL_ID     = "microsoft/Phi-3-mini-4k-instruct"   # was: "Qwen/Qwen2.5-1.5B-Instruct"
# --------------------------------------------------------

ADAPTER_REPO_ID   = "nooruiit-864/phi3-mini-roman-urdu-aiml-lora"  # separate repo so it doesn't overwrite the Qwen adapter
OUTPUT_DIR        = "./phi3-mini-roman-urdu-aiml-lora"
NUM_EPOCHS        = 3
# =====================


In [16]:
from datasets import load_dataset

if USE_LOCAL_FILE:
    # LOCAL_JSONL_PATH was set to just the filename, but the file is located in Google Drive.
    # The variable `dataset_path` already holds the correct full path.
    raw_ds = load_dataset("json", data_files=dataset_path, split="train")
else:
    raw_ds = load_dataset(DATASET_REPO_ID, split="train")

print(raw_ds)
print(raw_ds[0])
print(f"Total rows: {len(raw_ds)}")

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 200
})
{'messages': [{'role': 'system', 'content': 'Tum ek Hinglish AI/ML tutor ho jo hamesha pehle ek everyday analogy deta hai, phir uske baad technical explanation deta hai. Tumhara tone friendly aur encouraging hota hai.'}, {'role': 'user', 'content': 'TF-IDF kya hota hai aur ye simple word counting se better kyun hai?'}, {'role': 'assistant', 'content': "Socho tumhare paas 100 news articles hain aur tum dhoondna chahte ho ki kaunse words important hain. Agar tum sirf counting karo, to 'the', 'is', 'a' jaise common words sabse zyada baar aayenge — lekin woh useless hain. TF-IDF do cheezein combine karta hai: TF (Term Frequency) batata hai ek word document mein kitni baar aaya, aur IDF (Inverse Document Frequency) batata hai woh word kitna 'rare' hai across saare documents. Jo word ek document mein bohot use hua ho lekin baaki documents mein rare ho — wahi asal mein important hota hai. Isliye TF-IDF simple counting se zyada smart h

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

formatted_ds = raw_ds.map(to_chat_text, remove_columns=raw_ds.column_names)
print(formatted_ds[0]["text"])


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

<|system|>
Tum ek Hinglish AI/ML tutor ho jo hamesha pehle ek everyday analogy deta hai, phir uske baad technical explanation deta hai. Tumhara tone friendly aur encouraging hota hai.<|end|>
<|user|>
TF-IDF kya hota hai aur ye simple word counting se better kyun hai?<|end|>
<|assistant|>
Socho tumhare paas 100 news articles hain aur tum dhoondna chahte ho ki kaunse words important hain. Agar tum sirf counting karo, to 'the', 'is', 'a' jaise common words sabse zyada baar aayenge — lekin woh useless hain. TF-IDF do cheezein combine karta hai: TF (Term Frequency) batata hai ek word document mein kitni baar aaya, aur IDF (Inverse Document Frequency) batata hai woh word kitna 'rare' hai across saare documents. Jo word ek document mein bohot use hua ho lekin baaki documents mein rare ho — wahi asal mein important hota hai. Isliye TF-IDF simple counting se zyada smart hai.<|end|>
<|endoftext|>


In [18]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Pick one consistent compute dtype for the whole pipeline based on actual GPU support.
if torch.cuda.is_bf16_supported():
    COMPUTE_DTYPE = torch.bfloat16
    USE_FP16, USE_BF16 = False, True
else:
    COMPUTE_DTYPE = torch.float16
    USE_FP16, USE_BF16 = True, False
print(f"GPU: {torch.cuda.get_device_name(0)} | using dtype: {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    dtype=COMPUTE_DTYPE,
    device_map="auto",
    # trust_remote_code intentionally NOT set: both Qwen2.5 and Phi-3 are natively
    # supported by modern transformers. Setting it True pulls the model owner's custom
    # modeling code from the Hub, which can be stale and clash with the installed
    # transformers' config field names (e.g. `rope_scaling["type"]` vs `["rope_type"]`),
    # causing a KeyError at load time. Native support avoids that entirely.
)
model.config.use_cache = False  # required for training with gradient checkpointing


GPU: Tesla T4 | using dtype: torch.bfloat16


model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [20]:
from peft import LoraConfig, prepare_model_for_kbit_training
import bitsandbytes as bnb

model = prepare_model_for_kbit_training(model)

def find_target_modules(model):
    """Scan the model for 4-bit linear layer names to target with LoRA,
    excluding the output head (we don't want to adapt lm_head/embeddings)."""
    linear_cls = bnb.nn.Linear4bit
    names = set()
    for name, module in model.named_modules():
        if isinstance(module, linear_cls):
            leaf = name.split(".")[-1]
            if leaf not in ("lm_head",):
                names.add(leaf)
    return sorted(names)

target_modules = find_target_modules(model)
print(f"Auto-detected LoRA target modules for {BASE_MODEL_ID}: {target_modules}")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)


Auto-detected LoRA target modules for microsoft/Phi-3-mini-4k-instruct: ['down_proj', 'gate_up_proj', 'o_proj', 'qkv_proj']


In [21]:
import inspect
from trl import SFTTrainer, SFTConfig

# Compute a sensible warmup_steps fallback (~3% of total optimizer steps), since some
# trl versions don't accept `warmup_ratio` directly - if warmup_ratio IS accepted, it takes
# precedence anyway (HF's TrainingArguments rule: warmup_ratio > 0 overrides warmup_steps).
per_device_bs = 2
grad_accum = 4
steps_per_epoch = max(1, len(formatted_ds) // (per_device_bs * grad_accum))
total_steps = steps_per_epoch * NUM_EPOCHS
computed_warmup_steps = max(1, int(0.03 * total_steps))
print(f"~{steps_per_epoch} steps/epoch, {total_steps} total steps, warmup_steps={computed_warmup_steps}")

# Desired training settings (some keys may not exist in every trl version - that's OK,
# we filter below). Includes both old and new names for renamed args.
desired_args = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=per_device_bs,
    gradient_accumulation_steps=grad_accum,   # effective batch size = 8
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,               # used if this trl version supports it
    warmup_steps=computed_warmup_steps,  # fallback used if warmup_ratio isn't supported
    logging_steps=5,
    save_strategy="epoch",
    fp16=USE_FP16,    # matches the dtype chosen in Section 6 - avoids GradScaler dtype crash
    bf16=USE_BF16,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=512,              # older trl name
    max_length=512,                  # newer trl name
    packing=False,
)

valid_keys = set(inspect.signature(SFTConfig.__init__).parameters.keys())
filtered_args = {k: v for k, v in desired_args.items() if k in valid_keys}
skipped = set(desired_args) - set(filtered_args)
if skipped:
    print(f"Note: this trl version's SFTConfig doesn't accept these keys, skipping: {skipped}")
if "warmup_ratio" in skipped and "warmup_steps" in filtered_args:
    print(f"-> Using warmup_steps={computed_warmup_steps} as the warmup fallback instead.")

sft_config = SFTConfig(**filtered_args)
print(f"Final effective max sequence length setting: {getattr(sft_config, 'max_length', getattr(sft_config, 'max_seq_length', 'unknown'))}")


~25 steps/epoch, 75 total steps, warmup_steps=2
Note: this trl version's SFTConfig doesn't accept these keys, skipping: {'warmup_ratio', 'max_seq_length'}
-> Using warmup_steps=2 as the warmup fallback instead.
Final effective max sequence length setting: 512


In [22]:
trainer_kwargs = dict(
    model=model,
    args=sft_config,
    train_dataset=formatted_ds,
    peft_config=lora_config,
)

trainer_valid_keys = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
# trl renamed `tokenizer` -> `processing_class` in some versions - pass whichever is supported
if "processing_class" in trainer_valid_keys:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_valid_keys:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
trainer.train()


Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Step,Training Loss
5,3.492006
10,2.710320
15,2.354859
20,2.175296
25,2.078088
30,1.911485
35,1.811476
40,1.823487
45,1.745276
50,1.697251


TrainOutput(global_step=75, training_loss=1.974115130106608, metrics={'train_runtime': 2198.9958, 'train_samples_per_second': 0.273, 'train_steps_per_second': 0.034, 'total_flos': 3469662106214400.0, 'train_loss': 1.974115130106608, 'epoch': 3.0})

In [23]:
!pip freeze > requirements.txt
print("Saved requirements.txt with the exact working versions from this session.")


Saved requirements.txt with the exact working versions from this session.


In [24]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

trainer.model.push_to_hub(ADAPTER_REPO_ID)
tokenizer.push_to_hub(ADAPTER_REPO_ID)
print(f"Adapter pushed to: https://huggingface.co/{ADAPTER_REPO_ID}")


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  614kB / 50.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Adapter pushed to: https://huggingface.co/nooruiit-864/phi3-mini-roman-urdu-aiml-lora


In [25]:
EVAL_PROMPTS = [
    "Reinforcement learning kya hoti hai, simple lafzon mein samjhao?",
    "Exploration aur exploitation ka trade-off reinforcement learning mein kya hota hai?",
    "Supervised Fine-Tuning (SFT) aur RLHF mein practical farq kya hai?",
    "RLHF training pipeline ke main stages kya kya hote hain?",
    "Reward model RLHF mein kya role play karta hai?",
    "Post-finetuning steps mein kya kya hota hai (jaise merging, quantization, deployment)?",
    "LoRA adapter ko base model ke saath merge karna kab zaroori hota hai?",
    "Qwen model family kis liye jani jaati hai aur ye kaise alag hai doosre open models se?",
    "Instruction-tuned Qwen model, base Qwen model se kaise different hota hai?",
    "PEFT (Parameter-Efficient Fine-Tuning) kya hota hai, ek analogy se samjhao?",
    "PEFT ke different types (LoRA, prefix-tuning, adapters) mein kya farq hai?",
    "Catastrophic forgetting kya hota hai fine-tuning ke dauran?",
    "Model quantization deployment ke liye kyun zaroori hota hai?",
    "On-policy aur off-policy reinforcement learning algorithms mein kya farq hai?",
    "Policy gradient methods reinforcement learning mein kya karte hain?",
]
assert len(EVAL_PROMPTS) == 15, "Please provide exactly 15 held-out prompts."

# If your training rows used a consistent system prompt, put the SAME one here so the
# eval is fair (fine-tuned model was trained expecting this system context).
EVAL_SYSTEM_PROMPT = "Tum ek Hinglish AI/ML tutor ho jo hamesha pehle ek everyday analogy deta hai, phir uske baad technical explanation deta hai. Tumhara tone friendly aur encouraging hota hai."


In [26]:
import gc

# Free up memory from the training model before reloading for eval
del trainer, model
gc.collect()
torch.cuda.empty_cache()


In [27]:
from peft import PeftModel

def load_base():
    return AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb_config, dtype=COMPUTE_DTYPE, device_map="auto"
    )

def generate(model, prompt, max_new_tokens=200):
    messages = []
    if EVAL_SYSTEM_PROMPT:
        messages.append({"role": "system", "content": EVAL_SYSTEM_PROMPT})
    messages.append({"role": "user", "content": prompt})
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded.strip()

# --- Base model responses ---
base_model = load_base()
base_responses = [generate(base_model, p) for p in EVAL_PROMPTS]
del base_model
gc.collect(); torch.cuda.empty_cache()

# --- Fine-tuned (base + adapter) responses ---
ft_base = load_base()
ft_model = PeftModel.from_pretrained(ft_base, OUTPUT_DIR)  # or ADAPTER_REPO_ID to pull from Hub
ft_responses = [generate(ft_model, p) for p in EVAL_PROMPTS]
del ft_base, ft_model
gc.collect(); torch.cuda.empty_cache()


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [28]:
import pandas as pd

results = pd.DataFrame({
    "prompt": EVAL_PROMPTS,
    "base_response": base_responses,
    "finetuned_response": ft_responses,
    "base_task_relevance_1to5": [None] * 15,
    "base_tone_match_1to5": [None] * 15,
    "ft_task_relevance_1to5": [None] * 15,
    "ft_tone_match_1to5": [None] * 15,
})

pd.set_option("display.max_colwidth", None)
results.to_csv("day29_eval_results.csv", index=False)
results


,prompt,base_response,finetuned_response,base_task_relevance_1to5,base_tone_match_1to5,ft_task_relevance_1to5,ft_tone_match_1to5
0,"Reinforcement learning kya hoti hai, simple lafzon mein samjhao?","Reinforcement Learning (RL) ek technology hai, kitna bada logik hai, aur kisi ko ek dikhwaane kuch saath nadeekhna hai ki kisi ko log kisi ki ke saath likhaat samjhan aur log ki log kisi ko likhne ki kisi ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne ko log kisi ki likhne","Socho reinforcement learning ek trial-and-error game hai — agent (model) ek environment (real-world task) mein pehle tak randomly taki 'actions' (choices) bhi bhi make, ke sabse bade bhi sabse badi outcome banaye — good (reward) aur bad (punishment) — training ke liye. Agent ek self-coaching mein aata hai, lekin humnein reward system (jaise reward increase ho, pahunchi goal achieve karke) kyunki tak self-motivation deta hai, aur ye behavior iteratively optimize hota hai jab bade environment mein.",None,None,None,None
1,Exploration aur exploitation ka trade-off reinforcement learning mein kya hota hai?,"Ek Hinglish AI/ML tutor kijiye, reinforcement learning mein kiska reinforcement ka ek simple everyday analogy mein hoga.\n\nImagine yeh aabta kaha hai, ek big shop ke ek big kaha hai. Ek shop ke ek big kaam hai, ek big collection ke ek collection ho jana, aur ek big bhi ek big kaam hai, ek collection ke ek collection ho jana. Ek shop ke ek big kaam hai, ek collection ke ek collection ho jana, aur ek shop ke ek big kaam hai, ek collection ke ek collection ho jana. Ek shop ke ek big kaam hai, ek collection ke ek collection ho jana, aur ek shop ke ek big kaam hai, ek collection ke ek collection ho jana.\n\nEk big kaam hai, ek collection ke ek collection ho jana,","Socho exploration ek zyada risky jaisa hota hai — ek tum ek brand new restaurant ka baar baar dhoondne, isse usse bhi kam chalte hain, sabse kuch sawaal chalane. Exploitation ek zyada safe jaisa hota hai — ek tum ek familiar restaurant ka baar baar dhoondne, isse usse alag chalein, kam sawaal chalane. Reinforcement learning mein agar model ka learning curve nahi rakhna, isliye ek zyada exploration (random, risky choices) dikhaya jaata hai. Isliye reinforcement learning mein ek balance ke saath exploration aur exploitation mili jaata hai — ek bohot zyada exploration bhi sabse zyada exploitation dikhawa jaata hai.",None,None,None,None
2,Supervised Fine-Tuning (SFT) aur RLHF mein practical farq kya hai?,"Supervised Fine-Tuning (SFT) and Reinforcement Learning from Human Feedback (RLHF) are two advanced machine learning techniques used to improve the performance of AI models, particularly in natural language processing (NLP) and dialogue systems. Let's break down these concepts with everyday analogies and then dive into the technical explanations.\n\n### Everyday Analogy for SFT and RLHF\n\nImagine you're a chef who has created a basic recipe for a delicious dish. This recipe (your initial AI model) is good, but it might not be perfect for every taste. You decide to get feedback from your customers (the end-users or the environment in machine learning terms) to make it even better.\n\n1. **Supervised Fine-Tuning (SFT):** This is like a chef asking a few customers for feedback on specific aspects of the dish","Socho ye ek student mein bohot familiar question bhi hai (jaise original dataset ya already trained model ko), lekin sirf ye familiar context mein, aur sirf sirf answers ko kitna zaroori hai. RLHF pe us kuch bhi kuch extra questions/tasks banata hai jo original model ko ek context-sensitive training mein chahiye ho. SFT ek basic, focused training hai — sirf original context mein ko questions ko answers ko koi check karne ki bhi